In [1]:
import requests
import numpy as np
import lifesim

In [ ]:
# ---------- Set-Up ----------

# create bus
bus = lifesim.Bus()

# setting the options
bus.data.options.set_scenario('baseline')

# set options manually
bus.data.options.set_manual(diameter=4.)
bus.data.options.set_manual(output_path='/home/kirschkobold/LIFEsim/kira_testing')
bus.data.options.set_manual(output_filename='test_run')

In [3]:
# ---------- Downloading the P-Pop catalog ----------

# data = requests.get('https://raw.githubusercontent.com/kammerje/P-pop/main/TestPlanetPopulation.txt')

# with open('path/ppop_catalog.txt', 'wb') as file:
#     file.write(data.content)

In [ ]:
# ---------- Loading the Catalog ----------

bus.data.catalog_from_ppop(input_path='/home/kirschkobold/LIFEsim/kira_testing/ppop_catalog.txt', overwrite=True)
bus.data.catalog_remove_distance(stype='A', mode='larger', dist=0.)  # remove all A stars
bus.data.catalog_remove_distance(stype='M', mode='larger', dist=10.)  # remove M stars > 10pc to
# speed up calculation

Processed line 33048 of 33048


In [7]:
# ---------- Creating the Instrument ----------

# create modules and add to bus
instrument = lifesim.Instrument(name='inst')
bus.add_module(instrument)

transm = lifesim.TransmissionMap(name='transm')
bus.add_module(transm)

exo = lifesim.PhotonNoiseExozodi(name='exo')
bus.add_module(exo)
local = lifesim.PhotonNoiseLocalzodi(name='local')
bus.add_module(local)
star = lifesim.PhotonNoiseStar(name='star')
bus.add_module(star)

# connect all modules
bus.connect(('inst', 'transm'))
bus.connect(('inst', 'exo'))
bus.connect(('inst', 'local'))
bus.connect(('inst', 'star'))

bus.connect(('star', 'transm'))

In [8]:
# ---------- Creating the Optimizer ----------
# After every planet is given an SNR, we want to distribute the time available in the search phase
# such that we maximize the number of detections.

# optimizing the result
opt = lifesim.Optimizer(name='opt')
bus.add_module(opt)
ahgs = lifesim.AhgsModule(name='ahgs')
bus.add_module(ahgs)

bus.connect(('transm', 'opt'))
bus.connect(('inst', 'opt'))
bus.connect(('opt', 'ahgs'))

In [1]:
# ---------- Running the Simulation ----------

# run simulation. This function assigns every planet an SNR for 1 hour of integration time. Since
# we are currently only simulating photon noise, the SNR will scale with the integration time as
# sqrt(t)
instrument.get_snr()

opt.ahgs()

NameError: name 'instrument' is not defined

In [21]:
# ---------- Saving the Results ----------

bus.save()

Saving database and config files...
Catalog saved
[Done]


In [ ]:
# ---------- Reading the Results ----------
# import a previously saved catalog
bus_read = lifesim.Bus()
bus_read.build_from_config('/home/kirschkobold/LIFEsim/kira_testing/test_run.yaml')
bus_read.data.import_catalog(input_path='/home/kirschkobold/LIFEsim/kira_testing/test_run_catalog.hdf5')

In [30]:
mask_mtype = bus.data.catalog.stype == 'M'
mask = np.logical_and.reduce((bus.data.catalog.detected, bus.data.catalog.habitable, mask_mtype))
result_number = mask.sum()/500
print(result_number)

0.504


In [35]:
print(bus.data.catalog['habitable'].value_counts())
print(bus.data.catalog['detected'].value_counts())

habitable
False    6662
True     1115
Name: count, dtype: int64
detected
False    4710
True     3067
Name: count, dtype: int64
